In [1]:
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import pandas as pd


def select_device():
    if not torch.backends.mps.is_available():
        if not torch.backends.mps.is_built():
            print(
                "MPS not available because the current PyTorch install was not built with MPS enabled.")
        else:
            print(
                "MPS not available because the current MacOS version is not 12.3+ or there is no MPS-enabled device.")

        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        return torch.device("mps")


device = select_device()

In [2]:
class SlidingWindowUCBAgent:
    """
    Agent implementing Sliding Window UCB algorithm for action selection.

    Attributes
    ----------
    window_size : int
        The size of the sliding window for recent rewards and counts.
    """
    def __init__(self, window_size=1000):
        self.counts = None
        self.values = None
        self.c = 3
        self.window_size = window_size
        self.recent_rewards = None
        self.recent_counts = None
        self.recent_rewards_sum = None
        self.recent_counts_sum = None
        self.total_time_steps = 0

    def initialize(self, n_actions):
        self.counts = np.zeros(n_actions)
        self.values = np.zeros(n_actions)
        self.recent_rewards = [deque(maxlen=self.window_size) for _ in range(n_actions)]
        self.recent_counts = [deque(maxlen=self.window_size) for _ in range(n_actions)]
        self.recent_rewards_sum = np.zeros(n_actions)
        self.recent_counts_sum = np.zeros(n_actions)

    def get_action(self):
        if self.counts.min() == 0:
            idx = np.random.choice(np.where(self.counts == 0)[0])
            action = np.zeros(len(self.values))
            action[idx] = 1
        else:
            min_time_steps = min(self.total_time_steps, self.window_size)
            recent_values = self.recent_rewards_sum / self.recent_counts_sum
            ucb_values = recent_values + self.c * np.sqrt(
                2 * np.log(min_time_steps) / self.recent_counts_sum)
            action = ucb_values
        return action

    def update(self, actions, state):
        self.total_time_steps += 1
        for i, reward in enumerate(state):
            if reward >= 0:
                self.counts[i] += 1

                if len(self.recent_rewards[i]) == self.window_size:
                    self.recent_rewards_sum[i] -= self.recent_rewards[i][0]
                    self.recent_counts_sum[i] -= self.recent_counts[i][0]

                self.recent_rewards[i].append(reward)
                self.recent_counts[i].append(1)
                self.recent_rewards_sum[i] += reward
                self.recent_counts_sum[i] += 1
            else:
                self.counts[i] += 0
                self.recent_rewards[i].append(0)
                self.recent_counts[i].append(0)


In [3]:
class ROFARS_v1:
    def __init__(self, length=3600*12, n_camera=10, budget_ratio=0.5, data_path='data/train_test.txt'):
        # 43,200 seconds = 12 hours = 1/2 day
        self.length = length
        # 10 cameras by default
        self.n_camera = n_camera
        # set ratio
        self.budget_ratio = budget_ratio
        # generate cameras' stream
        self.train_cameras, self.test_cameras = self.init_cameras(data_path)
        self.reset()

    def reset(self, mode='train'):
        # train/test mode
        if mode == 'train':
            self.cameras = self.train_cameras
        elif mode == 'test':
            self.cameras = self.test_cameras
        self.index = 0
        self.rewards = []

    def init_cameras(self, path):
        data = pd.read_csv(path, sep=' ', names=['count', 'date', 'time'])
        assert self.length * self.n_camera * 2 <= len(data)
        train_cameras, test_cameras = [], []
        # assign the first half to training data
        train_index = range(self.n_camera)
        # assign the second half to testing data
        test_index = range(self.n_camera, self.n_camera * 2)
        for i in train_index:
            train_cameras.append(data[self.length*i:self.length*(i+1)].reset_index(drop=True))
        for i in test_index:
            test_cameras.append(data[self.length*i:self.length*(i+1)].reset_index(drop=True))
        return train_cameras, test_cameras

    def get_total_reward(self):
        return np.mean(self.rewards).round(3)

    def step(self, action):
        assert len(action) == self.n_camera
        # adding small noise to the action for randomness when all values are the same
        action += np.random.rand(*action.shape)/100
        # action is a vector of scores with n_camera-dimension
        # -1 refers to the unchecked camera state
        state = np.ones(self.n_camera)*(-1)
        # get the index of cameras for check according to the budget
        check_index = np.argsort(action)[::-1][:int(self.budget_ratio * self.n_camera)]
        for i in check_index:
            state[i] = self.cameras[i]['count'][self.index]

        # collect number of total faces as the reward
        reward = state[state != -1].sum()/self.n_camera
        self.rewards.append(reward)

        # check if stop
        if self.index == self.length-1:
            stop = True
        else:
            stop = False

        # step
        self.index += 1

        return reward, state, stop

In [4]:
"""
RNNtest script for 'Resource Optimization for Facial Recognition Systems (ROFARS)' project
author: Jasper Bruin @ UvA-MNS
date: 23/02/2023
"""
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time


def SWUCBExperiment():
    np.random.seed(0)
    env = ROFARS_v1()
    max_window_size = 100
    best_window_size = 1
    best_reward = -np.inf

    window_sizes = []
    total_rewards = []

    # Find the best sliding window in the training session
    for window_size in range(1, max_window_size + 1):
        agent = SlidingWindowUCBAgent(window_size=window_size * 60)
        agent.initialize(env.n_camera)

        # Training loop
        env.reset(mode='train')

        for t in tqdm(range(env.length), initial=2):
            action = agent.get_action()
            reward, state, stop = env.step(action)

            # Update the UCB Agent
            agent.update(action, state)

            if stop:
                break

        total_reward = env.get_total_reward()
        print(f'=== TRAINING === window size: {window_size}')
        print('[total reward]:', total_reward)

        # Save the best window size and total reward
        if total_reward > best_reward:
            best_reward = total_reward
            best_window_size = window_size

        # Record the window size and its total reward
        window_sizes.append(window_size)
        total_rewards.append(total_reward)

    # Use the best sliding window for testing
    agent = SlidingWindowUCBAgent(window_size=best_window_size * 60)
    agent.initialize(env.n_camera)
    env.reset(mode='test')

    for t in tqdm(range(env.length), initial=2):
        action = agent.get_action()
        reward, state, stop = env.step(action)

        # Update the UCB Agent
        agent.update(action, state)

        if stop:
            break

    test_total_reward = env.get_total_reward()
    print(f'====== TESTING window size ======')
    print('[total reward]:', test_total_reward)
    print(f'Best window size: {best_window_size}')
    print(f'Best [total reward]: {best_reward}')

    # Plot the window size and its total reward
    plt.plot(window_sizes, total_rewards,
             label=f"Best window size: {best_window_size}, Total reward: {best_reward:.3f}")
    plt.xlabel('Window Size', fontsize=12)
    plt.ylabel('Total Reward', fontsize=12)
    plt.title('Sliding Window UCB: Window Size vs Total Reward', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid()
    plt.tight_layout()
    plt.savefig('UCB.png')
    plt.show()


In [5]:
SWUCBExperiment()

FileNotFoundError: [Errno 2] No such file or directory: 'data/train_test.txt'